# huggingface accelerate, single-node distributed

spending some time on `accelerate` because i want one training loop that works on cpu, single-gpu, and ddp without rewriting. notes from getting it to launch.

In [ ]:
# pip install accelerate==0.12.0 transformers==4.21
import accelerate
print(accelerate.__version__)


In [ ]:
from accelerate import Accelerator
import torch, torch.nn as nn

accel = Accelerator()
model = nn.Linear(16, 4)
opt = torch.optim.SGD(model.parameters(), lr=1e-2)

# fake data
xs = torch.randn(64, 16)
ys = torch.randint(0, 4, (64,))
loader = torch.utils.data.DataLoader(list(zip(xs, ys)), batch_size=8)

model, opt, loader = accel.prepare(model, opt, loader)


In [ ]:
loss_fn = nn.CrossEntropyLoss()
for x, y in loader:
    out = model(x)
    loss = loss_fn(out, y)
    accel.backward(loss)
    opt.step(); opt.zero_grad()
print('done')


## launching

```
accelerate config
accelerate launch train.py
```

the config wizard is the thing that finally clicked it for me, before that i kept fighting torch.distributed envs.

diff vs last week: small. mostly cleanup.

In [ ]:
# print param count
# from utils import count_params
# print(count_params(model))


rerun on torch 1.12 cpu, same numbers within noise.